# ⛅ Weather-Advisory Support Bot
### Interactive Manual Testing & Evaluation Notebook (Google Colab & Local Jupyter)

This notebook provides an end-to-end interactive testbed and formal verification suite for the **Weather-Advisory Support Bot**.

**Key Architectural Guarantees:**
1. **Real LangGraph Directed Graph:** Multi-node graph with conditional branching (not a single prompt-and-response chain).
2. **100% Policy Grounding:** Every safety recommendation maps strictly to an approved Standard Operating Procedure (SOP) from `sops.yaml` cited by ID.
3. **No Hallucinated Weather Values:** All numbers in answers are grounded strictly in live Open-Meteo forecast API responses and verified via a deterministic guardrail.
4. **Deterministic Conflict Resolution:** When multiple SOPs match, one primary leads, secondary policies are surfaced with full advice, and the system explicitly explains **why** the primary rule was selected.
5. **Zero-Code Policy Updates:** SOPs are loaded fresh on every turn. An 11th/17th SOP can be added live without restarting or touching control-flow code.


In [1]:
# Setup & Dependency Installation
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab environment.")
    get_ipython().system("pip install -q langgraph langchain langchain-google-genai google-genai requests pyyaml pydantic python-dotenv")
except ImportError:
    IN_COLAB = False
    print("Running in local environment.")

import os
import sys
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Ensure current working directory is in path
sys.path.insert(0, os.path.abspath('.'))

load_dotenv()

# If in Colab and GOOGLE_API_KEY is not in .env, check Colab Secrets
if IN_COLAB and not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        colab_key = userdata.get("GOOGLE_API_KEY")
        if colab_key:
            os.environ["GOOGLE_API_KEY"] = colab_key
            print("Loaded GOOGLE_API_KEY from Google Colab Secrets.")
    except Exception:
        pass

# Ensure default model settings
if not os.environ.get("MODEL_PROVIDER"):
    os.environ["MODEL_PROVIDER"] = "google_genai"
if not os.environ.get("MODEL"):
    os.environ["MODEL"] = "gemini-3.5-flash-lite"

print("Active Model Provider:", os.environ.get("MODEL_PROVIDER"))
print("Active Model:", os.environ.get("MODEL"))


Running in local environment.
Active Model Provider: google_genai
Active Model: gemini-3.5-flash-lite


## 1. Graph Architecture Inspection
Inspect the nodes, conditional edges, and execution flow of the compiled LangGraph agent.


In [1]:
import graph

print('=== COMPILED LANGGRAPH NODES ===')
for node_name in graph.bot_graph.nodes:
    print(f'  [Node] {node_name}')

print('\nConditional branching sits at:')
print('  - parse_intent     -> (resolve_location | not_in_scope_reply)')
print('  - resolve_location -> (fetch_weather | error_reply)')
print('  - fetch_weather    -> (match_sops | error_reply)')
print('  - match_sops       -> (resolve_conflicts | no_match_reply)')
print('  - resolve_conflicts-> compose_answer -> verify_answer -> END')


=== COMPILED LANGGRAPH NODES ===
  [Node] __start__
  [Node] parse_intent
  [Node] resolve_location
  [Node] fetch_weather
  [Node] match_sops
  [Node] resolve_conflicts
  [Node] compose_answer
  [Node] verify_answer
  [Node] no_match_reply
  [Node] error_reply
  [Node] not_in_scope_reply

Conditional branching sits at:
  - parse_intent     -> (resolve_location | not_in_scope_reply)
  - resolve_location -> (fetch_weather | error_reply)
  - fetch_weather    -> (match_sops | error_reply)
  - match_sops       -> (resolve_conflicts | no_match_reply)
  - resolve_conflicts-> compose_answer -> verify_answer -> END


## 2. Interactive Manual Query Sandbox
Test any question live with full traceability. Change the `user_query` below and run the cell!


In [2]:
import graph

def test_interactive_query(query: str):
    print(f'User Query: "{query}"\n' + '-'*60)
    res = graph.ask_bot(query)
    
    print('BOT REPLY:')
    print(res.get('reply'))
    print('\n' + '='*60)
    print('AUDIT TRACEABILITY:')
    chosen = res.get('chosen_sop')
    if chosen:
        print(f"  * Primary SOP:     {chosen.get('id')} - {chosen.get('title')} (Severity {chosen.get('severity')})")
    else:
        print("  * Primary SOP:     None (Honest boundary refusal)")
        
    if res.get('selection_reason'):
        print(f"  * Selection Reason: {res.get('selection_reason')}")
        
    others = res.get('other_sops', [])
    if others:
        sec_ids = ', '.join([s['id'] for s in others])
        print(f"  * Secondary SOPs:  {sec_ids}")
        
    facts = res.get('weather_facts')
    if facts:
        print(f"  * Verified Facts:  Temp: {facts.get('temperature_2m')} C | Wind: {facts.get('wind_speed_10m')} km/h | Gusts: {facts.get('wind_gusts_10m')} km/h | Rain: {facts.get('precipitation')} mm ({facts.get('precipitation_probability')}%) | UV: {facts.get('uv_index')}")

# Test 1: Travel in Bhopal
test_interactive_query("is this the perfect weather traveling in Bhopal")


User Query: "is this the perfect weather traveling in Bhopal"
------------------------------------------------------------
BOT REPLY:
Based on our safety guidelines under **SOP-015**, travel conditions for your route in Bhopal, India today are favorable. The rain chance is 0.0%, the wind speed is 5.9 km/h, and visibility is clear. There are currently no active travel hazard alerts or road slickness warnings in effect, so please simply observe standard driving safety and speed regulations.

Additionally, under **SOP-013**, current weather conditions (including a temperature of 28.7°C, wind speed of 5.9 km/h, peak gusts of 20.2 km/h, 0.0% rain chance, and 0.0 mm daily rain sum) fall within our calm weather baseline with no warnings triggered. Please remember to check local conditions before departing.

*Primary Selection Rationale: Clearance policy [SOP-015] was selected as primary because current weather metrics confirm favorable, safe conditions specifically for travel.*

AUDIT TRACEAB

In [3]:
# Test 2: Cycling in Bhopal today
import graph
if 'test_interactive_query' not in globals():
    from graph import ask_bot
    res = graph.ask_bot("Is it safe to cycle in Bhopal today?")
    print("BOT REPLY:\n", res.get('reply'))
else:
    test_interactive_query("Is it safe to cycle in Bhopal today?")


User Query: "Is it safe to cycle in Bhopal today?"
------------------------------------------------------------
BOT REPLY:
Hello! Here is your outdoor safety advisory based on our official company procedures:

According to **SOP-013**, no policy warnings are triggered for cycling in Bhopal, India today, as the current weather (temperature: 28.7°C, wind: 5.9 km/h, peak gusts: 20.2 km/h, rain chance: 0.0%, daily rain sum: 0.0 mm) remains within calm baseline thresholds. Please note that this is not a guarantee of safety, so you should always check local conditions before departing.

Additionally, under **SOP-015**, travel conditions are favorable for your route today (rain chance: 0.0%, wind speed: 5.9 km/h, visibility clear) with no active travel hazard alerts or road slickness warnings in effect. Please continue to observe standard safety and speed regulations.

*Primary Selection Rationale: Baseline informational rule [SOP-013] was selected as primary because current weather metrics a

In [4]:
# Test 3: Uncovered Activity (Honest Boundary Refusal)
import graph
if 'test_interactive_query' not in globals():
    res = graph.ask_bot("Is it safe to go scuba diving in the Andamans?")
    print("BOT REPLY:\n", res.get('reply'))
else:
    test_interactive_query("Is it safe to go scuba diving in the Andamans?")


User Query: "Is it safe to go scuba diving in the Andamans?"
------------------------------------------------------------
BOT REPLY:
I don't have guidance for that. Our standard operating procedures cover outdoor exercise (cycling, running, hiking), travel and commuting safety, vulnerable groups (children, elderly, pets in heat/cold), and general outdoor leisure (such as picnics). I only provide recommendations when an approved policy applies.

AUDIT TRACEABILITY:
  * Primary SOP:     None (Honest boundary refusal)
  * Selection Reason: Baseline informational rule [SOP-013] was selected as primary because current weather metrics are within calm operational thresholds for cycling.
  * Verified Facts:  Temp: 27.9 C | Wind: 12.2 km/h | Gusts: 33.1 km/h | Rain: 0.0 mm (66.0%) | UV: 0.9


## 3. Systematic Verification of Assignment Test Cases
Here we execute each of the required test scenarios defined in the assignment specification.


In [5]:
# Scenario 1: High Wind & Two-Wheeler / Cycling Safety (SOP-001)
import graph
import weather

mock_wind_facts = {
    "temperature_2m": 25.0,
    "apparent_temperature": 25.0,
    "wind_speed_10m": 38.0,
    "wind_gusts_10m": 52.0,
    "precipitation": 0.0,
    "precipitation_probability": 0.0,
    "uv_index": 3.0,
    "weather_code": 1,
    "pressure_msl": 1010.0,
    "relative_humidity_2m": 40.0,
    "daily_precipitation_sum": 0.0,
    "precipitation_probability_max": 0.0,
    "wind_gusts_10m_max": 52.0,
    "uv_index_max": 3.0,
    "time": "2026-11-13T12:00",
    "hour": 12,
    "location_name": "Bhopal, India",
}

orig_extract = weather.extract_weather_facts
try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_wind_facts)
    res = graph.ask_bot("Is it safe to ride my scooter in Bhopal?")
    print("Query: Is it safe to ride my scooter in Bhopal?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


Query: Is it safe to ride my scooter in Bhopal?
Chosen SOP: SOP-001
Reply: Based on official safety guidelines under **SOP-001**, a high wind safety warning is currently in effect for Bhopal, India, with sustained winds of 38.0 km/h and peak gusts reaching 52.0 km/h. 

Riding a bicycle, scooter, or motorcycle in these weather conditions creates extreme lateral drift, instability near heavy vehicles, and collision hazards from flying debris. 

You must avoid two-wheeled transit; please postpone your trip or use enclosed 4-wheel transport instead.

*Primary Selection Rationale: Policy [SOP-001] was selected as primary because it has the highest hazard severity (Severity 3) among all 1 matching policies.*


In [6]:
# Scenario 2: Peak UV Index Midday Rule (SOP-002)
import graph
import weather

mock_uv_facts = {
    "temperature_2m": 32.0,
    "apparent_temperature": 34.0,
    "wind_speed_10m": 12.0,
    "wind_gusts_10m": 18.0,
    "precipitation": 0.0,
    "precipitation_probability": 0.0,
    "uv_index": 9.5,
    "weather_code": 0,
    "pressure_msl": 1012.0,
    "relative_humidity_2m": 35.0,
    "daily_precipitation_sum": 0.0,
    "precipitation_probability_max": 0.0,
    "wind_gusts_10m_max": 18.0,
    "uv_index_max": 9.5,
    "time": "2026-11-13T13:00",
    "hour": 13,
    "location_name": "Delhi, India",
}

orig_extract = weather.extract_weather_facts
try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_uv_facts)
    res = graph.ask_bot("Should I go for a run at noon in Delhi?")
    print("Query: Should I go for a run at noon in Delhi?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


Query: Should I go for a run at noon in Delhi?
Chosen SOP: SOP-002
Reply: Based on our safety guidelines under **SOP-002**, there is a very high UV radiation alert with a UV index of 9.5 during peak solar hours from 11:00 to 16:00. Direct sun exposure during this window carries a rapid risk of acute sunburn, photokeratitis, and cellular skin damage. 

To stay safe, avoid unprotected strenuous outdoor exercise during this midday window. You should reschedule your workouts to the early morning or after 16:00. If you must be outside, seek shade, wear UV-protective clothing, and apply broad-spectrum SPF 50+ sunscreen.

Additionally, please note our supplementary guidance under **SOP-013**, which notes that current weather conditions in Delhi, India feature a temperature of 32.0°C, a wind speed of 12.0 km/h, peak gusts of 18.0 km/h, a rain chance of 0.0%, and a daily rain sum of 0.0 mm. While these specific metrics fall within calm baseline thresholds (Severity 1), this does not guarantee s

In [7]:
# Scenario 3: Monsoon Heavy Rain Depression Proxy (SOP-SYSTEM-RAIN)
import graph
import weather

mock_monsoon_facts = {
    "temperature_2m": 22.0,
    "apparent_temperature": 22.0,
    "wind_speed_10m": 30.0,
    "wind_gusts_10m": 45.0,
    "precipitation": 15.0,
    "precipitation_probability": 95.0,
    "uv_index": 1.0,
    "weather_code": 65,
    "pressure_msl": 995.0,
    "relative_humidity_2m": 98.0,
    "daily_precipitation_sum": 85.0,
    "precipitation_probability_max": 100.0,
    "wind_gusts_10m_max": 50.0,
    "uv_index_max": 2.0,
    "time": "2026-11-13T10:00",
    "hour": 10,
    "location_name": "Bhopal, India",
}

orig_extract = weather.extract_weather_facts
try:
    weather.extract_weather_facts = lambda *args, **kwargs: dict(mock_monsoon_facts)
    res = graph.ask_bot("Is it safe to drive to work in Bhopal today?")
    print("Query: Is it safe to drive to work in Bhopal today?")
    print("Chosen SOP (Global Override):", res.get('chosen_sop', {}).get('id'))
    print("Selection Reason:", res.get('selection_reason'))
    print("Reply:", res.get('reply'))
finally:
    weather.extract_weather_facts = orig_extract


Query: Is it safe to drive to work in Bhopal today?
Chosen SOP (Global Override): SOP-SYSTEM-RAIN
Selection Reason: System override rule [SOP-SYSTEM-RAIN] takes absolute precedence over all other policies due to widespread severe hazard conditions (scope: global, Severity 4).
Reply: According to our active safety guidelines under **SOP-SYSTEM-RAIN**, a critical regional rain alert is in effect for Bhopal, India, with a forecast daily rainfall of 85.0 mm, a rain probability of 100.0%, and a surface pressure of 995.0 hPa. Due to expected flash flooding, severe waterlogging, and transport standstills, you must cancel all non-emergency outdoor movement immediately, stay indoors in secure structures, and monitor state disaster management warnings.

Additionally, please note the following active policies:
* **[SOP-001]**: High wind safety warning noting a sustained wind of 30.0 km/h and peak gusts of 45.0 km/h. Avoid two-wheeled transit like bicycles, scooters, or motorcycles due to extreme 

## 4. Multi-turn Session Memory Verification
Verify that location and activity context persists cleanly across turns using LangGraph MemorySaver checkpointer.


In [10]:
import time
import graph

thread_session = "colab_session_" + str(int(time.time()))

# Turn 1: Specify location and activity
res1 = graph.ask_bot("Is it safe to cycle in Bhopal?", thread_id=thread_session)
print("Turn 1 Reply:", res1.get('reply')[:120], "...")
print("Turn 1 Resolved Location:", res1.get('location'))

# Turn 2: Follow-up query without mentioning location
res2 = graph.ask_bot("What about going for a run this evening?", thread_id=thread_session)
print("\nTurn 2 Reply:", res2.get('reply')[:120], "...")
print("Turn 2 Inherited Location:", res2.get('location'))
assert res2.get('location') == 'Bhopal, India' or 'Bhopal' in str(res2.get('location'))
print("\n Memory test PASSED: Location correctly persisted across turns!")


Turn 1 Reply: Based on our safety guidelines under **SOP-013**, there are no policy warnings triggered for cycling in Bhopal, India to ...
Turn 1 Resolved Location: Bhopal, India

Turn 2 Reply: Hello! Here is your outdoor safety advisory based on our official company procedures:

According to **SOP-013**, no poli ...
Turn 2 Inherited Location: Bhopal, India

 Memory test PASSED: Location correctly persisted across turns!


## 5. Live Dynamic SOP Hot-Reload Test
Demonstrates adding a new 17th SOP live to `sops.yaml` without touching control-flow code!


In [9]:
import yaml
from pathlib import Path
import graph
import weather

sops_path = Path('sops.yaml')
with open(sops_path, 'r', encoding='utf-8') as f:
    orig_sops = yaml.safe_load(f)

new_test_sop = {
    "id": "SOP-999",
    "title": "Extreme Monsoon Flash Flood Warning",
    "category": "travel",
    "severity": 4,
    "type": "numeric",
    "applies_to": ["travel", "driving", "commute"],
    "condition": "daily_precipitation_sum >= 1.0",
    "advice": "URGENT: Flash flood protocol active in {location_name}. Avoid travel completely. Policy: SOP-999.",
    "rationale": "Extreme flood hazard to motorists.",
}

orig_extract = weather.extract_weather_facts
try:
    with open(sops_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(orig_sops + [new_test_sop], f)
    print("Temporarily appended SOP-999 to sops.yaml...")
    
    mock_rain = {
        "temperature_2m": 22.0,
        "apparent_temperature": 22.0,
        "wind_speed_10m": 10.0,
        "wind_gusts_10m": 15.0,
        "precipitation": 5.0,
        "precipitation_probability": 90.0,
        "uv_index": 1.0,
        "weather_code": 65,
        "pressure_msl": 1005.0,
        "relative_humidity_2m": 90.0,
        "daily_precipitation_sum": 5.0,
        "precipitation_probability_max": 90.0,
        "wind_gusts_10m_max": 15.0,
        "uv_index_max": 2.0,
        "time": "2026-11-13T10:00",
        "hour": 10,
        "location_name": "Bhopal, India",
    }
    weather.extract_weather_facts = lambda *args, **kwargs: mock_rain
    
    res = graph.ask_bot("Can I drive to work in Bhopal today?")
    print("Chosen SOP:", res.get('chosen_sop', {}).get('id'))
    assert res.get('chosen_sop', {}).get('id') == "SOP-999"
    print("\n Zero-Code Hot-Reload PASSED: New rule immediately enforced on the next turn!")
finally:
    with open(sops_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(orig_sops, f)
    weather.extract_weather_facts = orig_extract
    print("Restored original sops.yaml.")


NameError: name 'Path' is not defined

## 6. Automated Evaluation Suite Execution & Results Table
Run the complete automated evaluation suite and view the formatted summary DataFrame.


In [ ]:
import evals
import pandas as pd

print('Running full automated test suite...')
evals.EVAL_RESULTS.clear()
evals.run_all_evals()

df = pd.DataFrame(evals.EVAL_RESULTS)
df = df.rename(columns={
    'name': 'Test Case',
    'check': 'Target Evaluation',
    'criteria': 'Pass Criteria',
    'status': 'Status',
    'notes': 'Observation Notes'
})

print(f"\nTOTAL TESTS: {len(df)} | PASSED: {(df['Status'] == 'PASS').sum()} | FAILED: {(df['Status'] == 'FAIL').sum()}")
df[['Test Case', 'Status', 'Pass Criteria', 'Observation Notes']]
